## First and easy example (Mole Fraction)

As a first example we are going to obtain all the derivatives of a composition
only function.

The best candidate is the mole fraction

In [1]:
# You can also do: "import thermodiff as td", and access to all this things.
from thermodiff import idx_function    # Index based functions
from thermodiff import sum_components  # Sympy summation over component number
from thermodiff import sum_custom      # Sympy summation over custom index
from thermodiff import n               # Mole number vector
from thermodiff import T               # Temperature
from thermodiff import P               # Pressure
from thermodiff import V               # Volume
from thermodiff import R               # Ideal gas constant
from thermodiff import k, l, m         # indexes for summations
from thermodiff import DiffPlz         # Class that makes the derivatives

# We may still need sympy for some things.
import sympy as sp

# To display latext code in the notebook.
from IPython.display import display, Math

Mole fraction is an index base function of the mole number vector:

$$
    x_k = f(\textbf{n}) = \frac{n_k}{\sum_{l=1}^{N_c} n_l}
$$

Always use k, l, m for indexes in your expressions, since i and j are reserved
to differentiate.

Thermodiff provides an easy way of defining an indexbased function:

In [2]:
# index based function with lazy derivatives function of mole numbers n
xk = idx_function("x")(n[k])

xk

x(n[k])

Let see what "lazy derivative" mean:

In [3]:
diff = DiffPlz(xk, [xk], name="x")

In [4]:
diff.dni

Derivative(x(n[k]), n[i])

In [5]:
diff.dnidnj

Derivative(x(n[k]), n[i], n[j])

In [6]:
diff.dt

0

As you can see, index based function with their "lazy derivatives" only express
the derivative a symbolic representation that that function must be
differentiated. In the case of dt (temperature derivative), the result is 0 as
expected (you can check by yourself the volume and pressure derivatives dv and
p)

This thing is useful if you want to include functions in your expressions that
doesn't have an explicit functionality. Then later, you can obtain the
derivatives of those functions separately.

Before differentiating something, let's check what the cleaning method do:

In [7]:
latex = diff.latex_readable_plz()

display(Math(latex["dn_i"]))

<IPython.core.display.Math object>

Maybe it's ugly for you to have the explicit functionality of the functions.
Then you can use the clean method:

In [8]:
latex = diff.latex_readable_plz()

display(Math(latex["dn_i"]))

<IPython.core.display.Math object>

After that, let's obtain the true derivatives of the molar fraction. For that
we can use a shortcut of thermodiff for summations:

In [9]:
expr = n[k] / sum_components(n[l], l)

expr

n[k]/Sum(n[l], (l, 1, N_c))

Now we plz differentiate:

In [10]:
diff_plz = DiffPlz(expr, indexes=[k, l], name="x_k")

In [11]:
diff_plz.dni

Piecewise((-n[i]/Sum(n[l], (l, 1, N_c))**2 + 1/Sum(n[l], (l, 1, N_c)), Eq(i, k)), (-n[k]/Sum(n[l], (l, 1, N_c))**2, True))

In [12]:
diff_plz.dnidnj

Piecewise((2*n[i]/Sum(n[l], (l, 1, N_c))**3 - 2/Sum(n[l], (l, 1, N_c))**2, Eq(i, j) & Eq(i, k)), (2*n[i]/Sum(n[l], (l, 1, N_c))**3 - 1/Sum(n[l], (l, 1, N_c))**2, Eq(i, k)), (2*n[j]/Sum(n[l], (l, 1, N_c))**3 - 1/Sum(n[l], (l, 1, N_c))**2, Eq(j, k)), (2*n[k]/Sum(n[l], (l, 1, N_c))**3, True))

## Second example (summation)

Let's analyze what happens when we have a summation with index based functions.
Imagine you have a term like this:

$$
    \sum_{k=1}^{N_c} n_k \phi_k \tau_{lk}
$$

Being $\phi_k$ and $\tau_{lk}$ index based functions:

$$
    \phi_k = f(\textbf{n})
$$

something like the terms of UNIQUAC ($r_k$ is a constant parameters):

$$
    \phi_k = \frac{r_k n_k}{\sum_{l=1}^{N_c} n_l r_l}
$$

And $\tau_{lk}$ something like:

$$
    \tau_{lk} = g(T)
$$

But internally $l$ and $k$ are indexes that refer to the binary parameters
between components $l$ and $k$.

The difference between the functions is that one of them us a function of the
mole number vector and its index represent the component $k$.

The other is dependent only on temperature, but the indexes represent two
components. Since $\tau_{lk}$ is not a function of the mole number vector, its
derivative with respect to the mole number vector will be zero. For this, we 
have to construct this functions differently:

In [13]:
phi_k = idx_function(r"\phi")(n[k])

phi_k

\phi(n[k])

Check how I defined the function $\phi_k$. The symbol of the function is 
only $\phi$. Then I made it function of the mole number of the component $k$.
This way you can follow the subscript of the function by the subscript of the
argument. For that:

$$
    \phi_k = \phi(n_k)
$$

Later we are going to clean this representation.

In [14]:
tau_lk = idx_function(r"\tau")(l, k, T)

tau_lk

\tau(l, k, T)

Check out that the function $\tau_{lk}$ the symbol is only $\tau$. Then I made
it function of the temperature and the two indexes $l$ and $k$. This way you
can follow the subscript of the function by the subscript of the argument. But
in this case is not function of $n_l$ and $n_k$.

$$
    \tau_{lk}(T) = \tau(l, k, T)
$$

Now let's build our expression:

In [15]:
expr = sum_components(n[k] * phi_k * tau_lk, k)

expr

Sum(\phi(n[k])*\tau(l, k, T)*n[k], (k, 1, N_c))

Now we differentiate:

In [16]:
diff = DiffPlz(expr, internal_functions=[phi_k, tau_lk], indexes=[k, l], name="S")

In [17]:
diff.dni

\phi(n[i])*\tau(l, i, T) + Sum(\tau(l, k, T)*Derivative(\phi(n[k]), n[i])*n[k], (k, 1, N_c))

In [18]:
diff.dnidnj

\tau(l, i, T)*Derivative(\phi(n[i]), n[j]) + \tau(l, j, T)*Derivative(\phi(n[j]), n[i]) + Sum(\tau(l, k, T)*Derivative(\phi(n[k]), n[i], n[j])*n[k], (k, 1, N_c))

In [19]:
diff.dt

Sum(\phi(n[k])*Derivative(\tau(l, k, T), T)*n[k], (k, 1, N_c))

In [20]:
diff.dt2

Sum(\phi(n[k])*Derivative(\tau(l, k, T), (T, 2))*n[k], (k, 1, N_c))

In [21]:
diff.dtdni

\phi(n[i])*Derivative(\tau(l, i, T), T) + Sum(Derivative(\phi(n[k]), n[i])*Derivative(\tau(l, k, T), T)*n[k], (k, 1, N_c))

In [22]:
display(Math(diff.latex_readable_plz()["dn_i"]))

<IPython.core.display.Math object>

In [23]:
display(Math(diff.latex_readable_plz()["dn2"]))

<IPython.core.display.Math object>

In [24]:
display(Math(diff.latex_readable_plz()["dT"]))

<IPython.core.display.Math object>

In [25]:
display(Math(diff.latex_readable_plz()["dT2"]))

<IPython.core.display.Math object>

In [26]:
display(Math(diff.latex_readable_plz()["dTn"]))

<IPython.core.display.Math object>